<a href="https://colab.research.google.com/github/ErasmoR/Erasmor/blob/master/ProyectoFinalpowerbiV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación + Imports + Carpeta de salida

In [3]:
!pip install -q pandas matplotlib seaborn pyreadstat itables statsmodels openpyxl

import os, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat
import warnings
import statsmodels.api as sm
from scipy.stats import norm

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

OUT_DIR = "outputs_powerbi"
os.makedirs(OUT_DIR, exist_ok=True)

print("✅ Librerías listas y carpeta creada:", OUT_DIR)

✅ Librerías listas y carpeta creada: outputs_powerbi


Subir y cargar el archivo

In [9]:
from google.colab import files
uploaded = files.upload()

SAV_NAME = "240322base de datos completa.sav"  # ajusta si tu archivo tiene otro nombre exacto
df, meta = pyreadstat.read_sav(SAV_NAME)

print("Shape:", df.shape)
print("\nInformación general del dataset")
df.info()

# Corrección estrictamente obligatoria: indentación del for (en tu .py estaba roto)
for columna in df.columns:
    print(f"{columna}: {df[columna].dtype}")

KeyboardInterrupt: 

 Construcción de variables de hogar + DF_POP (df_modelo_final)

In [15]:
df_modelo = df.copy()
df = df.copy()

id_hogar = ['llave_sec', 'provincia', 'prov', 'unidad', 'cuest', 'hogar']

cols_hogar = ['ingreso_hogar', 'personas', 'niños', 'adultos_mayores', 'dependencia', 'ingreso_per_capita']
df_modelo = df_modelo.drop(columns=cols_hogar, errors='ignore')

df_modelo['niño'] = (df_modelo['Edad'] < 15).astype(int)
df_modelo['adulto_mayor'] = (df_modelo['Edad'] >= 65).astype(int)

df_modelo['Salario_efectivo_empleado'] = df_modelo['Salario_efectivo_empleado'].fillna(0)
df_modelo['Salario_efectivo_independiente'] = df_modelo['Salario_efectivo_independiente'].fillna(0)

df_modelo['ingreso_total_individual'] = (
    df_modelo['Salario_efectivo_empleado'] + df_modelo['Salario_efectivo_independiente']
)

hogar = df_modelo.groupby(id_hogar).agg(
    ingreso_hogar=('ingreso_total_individual', 'sum'),
    personas=('Edad', 'count'),
    niños=('niño', 'sum'),
    adultos_mayores=('adulto_mayor', 'sum')
).reset_index()

hogar['dependencia'] = (hogar['niños'] + hogar['adultos_mayores']) / hogar['personas']
hogar['ingreso_per_capita'] = hogar['ingreso_hogar'] / hogar['personas']

df_modelo = df_modelo.merge(hogar, on=id_hogar, how='left')

columnas_finales = id_hogar + [
    'Sexo','Edad','Jefe_de_hogar','Asiste_a_la_escuela','Grado_alcanzado','ocu_des',
    'Salario_efectivo_empleado','Salario_efectivo_independiente',
    'Recibe_jubilacion_pension','Uso_internet_últimos_6_meses',
    'ingreso_hogar','personas','niños','adultos_mayores','dependencia','ingreso_per_capita'
]

df_modelo_final = df_modelo[columnas_finales].copy()

print("✅ DF_POP (df_modelo_final) shape:", df_modelo_final.shape)
df_modelo_final.head()

✅ DF_POP (df_modelo_final) shape: (42925, 22)


,llave_sec,provincia,prov,unidad,cuest,hogar,Sexo,Edad,Jefe_de_hogar,Asiste_a_la_escuela,...,Salario_efectivo_empleado,Salario_efectivo_independiente,Recibe_jubilacion_pension,Uso_internet_últimos_6_meses,ingreso_hogar,personas,niños,adultos_mayores,dependencia,ingreso_per_capita
0,1.0,01,01,001,01,1,1.0,44.0,1.0,2.0,...,0.0,1516.0,0.0,2.0,2116.0,3,1,0,0.333333,705.333333
1,1.0,01,01,001,01,1,2.0,21.0,2.0,2.0,...,600.0,0.0,0.0,2.0,2116.0,3,1,0,0.333333,705.333333
2,1.0,01,01,001,01,1,1.0,4.0,3.0,2.0,...,0.0,0.0,0.0,NaN,2116.0,3,1,0,0.333333,705.333333
3,2.0,01,01,001,02,1,2.0,22.0,1.0,2.0,...,700.0,0.0,0.0,1.0,1550.0,3,1,0,0.333333,516.666667
4,2.0,01,01,001,02,1,1.0,26.0,2.0,2.0,...,850.0,0.0,0.0,1.0,1550.0,3,1,0,0.333333,516.666667


Recodificaciones mínimas + educacion_grupo

In [16]:
dicc_sexo = {1: 'Masculino', 2: 'Femenino'}
dicc_jefe_h = {
    1: 'Jefe o jefa', 2: 'Conyugue', 3: 'Hijo o hija',
    4: 'Otro pariente', 5: 'Servicio domestico'
}
dicc_provincia = {
    '01': 'Bocas del Toro','02': 'Coclé','03': 'Colón','04': 'Chiriquí','05': 'Darién',
    '06': 'Herrera','07': 'Los Santos','08': 'Panamá','09': 'Veraguas','10': 'Comarca Guna Yala',
    '11': 'Comarca Emberá-Wounaan','12': 'Comarca Ngäbe-Buglé','13': 'Panamá Oeste'
}

df_modelo_final.loc[:, 'Sexo'] = df_modelo_final['Sexo'].replace(dicc_sexo).astype(str)
df_modelo_final.loc[:, 'Jefe_de_hogar'] = df_modelo_final['Jefe_de_hogar'].replace(dicc_jefe_h).astype(str)

df_modelo_final['provincia'] = df_modelo_final['provincia'].astype(str).str.zfill(2)
df_modelo_final['provincia'] = df_modelo_final['provincia'].replace(dicc_provincia)

dicc_nivel_edu = {
    1:'Ningun grado',2:'Prekínder o prejardín',3:'Kínder o jardín',4:'Enseñanza especial',
    11:'Primaria',12:'Primaria',13:'Primaria',14:'Primaria',15:'Primaria',16:'Primaria',
    21:'Vocacional',22:'Vocacional',23:'Vocacional',
    31:'Primer ciclo premedia',32:'Primer ciclo premedia',33:'Primer ciclo premedia',
    34:'Segundo ciclo media',35:'Segundo ciclo media',36:'Segundo ciclo media',
    41:'Superior no universitaria',42:'Superior no universitaria',
    51:'Superior universitaria',52:'Superior universitaria',53:'Superior universitaria',
    54:'Superior universitaria',55:'Superior universitaria',56:'Superior universitaria',
    61:'Especialidad',71:'Maestría',72:'Maestría',81:'Doctorado',82:'Doctorado',83:'Doctorado',84:'Doctorado'
}

df_modelo_final['Grado_alcanzado'] = pd.to_numeric(df_modelo_final['Grado_alcanzado'], errors='coerce')
df_modelo_final['nivel_edu'] = df_modelo_final['Grado_alcanzado'].map(dicc_nivel_edu).astype('object')
df_modelo_final['nivel_edu'] = df_modelo_final['nivel_edu'].fillna('Menores de 4 años')

# ✅ Función faltante en tu .py (mínima y necesaria para que corra)
def clasificar_educacion(nivel):
    if pd.isna(nivel):
        return "Sin información"
    nivel = str(nivel).lower()
    if "primaria" in nivel or "kínder" in nivel or "prek" in nivel:
        return "Básica"
    if "premedia" in nivel or "media" in nivel or "vocacional" in nivel:
        return "Media"
    if "superior" in nivel or "universitaria" in nivel:
        return "Superior"
    if "maestr" in nivel or "doctor" in nivel or "especial" in nivel:
        return "Posgrado"
    return "Otro"

df_modelo_final['educacion_grupo'] = df_modelo_final['nivel_edu'].apply(clasificar_educacion)

print("✅ Recodificaciones listas")
df_modelo_final[['Sexo','provincia','nivel_edu','educacion_grupo']].head()


✅ Recodificaciones listas


,Sexo,provincia,nivel_edu,educacion_grupo
0,Masculino,Bocas del Toro,Superior universitaria,Superior
1,Femenino,Bocas del Toro,Segundo ciclo media,Media
2,Masculino,Bocas del Toro,Ningun grado,Otro
3,Femenino,Bocas del Toro,Segundo ciclo media,Media
4,Masculino,Bocas del Toro,Segundo ciclo media,Media


Modelo Mincer + DF_WAGE (df_mincer)

In [19]:
# ✅ Función faltante en tu .py: necesaria para construir anios_educacion
def calcular_anios_educacion(codigo):
    try:
        c = int(codigo)
    except:
        return np.nan
    if c in [1,2,3,4]:
        return 0
    if 11 <= c <= 16:
        return 6
    if 21 <= c <= 23:
        return 12
    if 31 <= c <= 36:
        return 12
    if 41 <= c <= 42:
        return 14
    if 51 <= c <= 56:
        return 16
    if c == 61:
        return 17
    if 71 <= c <= 72:
        return 18
    if 81 <= c <= 84:
        return 21
    return np.nan

df_mincer = df_modelo_final.copy()
df_mincer = df_mincer[df_mincer['ocu_des'] == 'Ocupados'].copy()

df_mincer['anios_educacion'] = df_mincer['Grado_alcanzado'].apply(
    lambda x: calcular_anios_educacion(x) if pd.notna(x) else np.nan
)

df_mincer['ingreso_laboral'] = (
    df_mincer['Salario_efectivo_empleado'].fillna(0) +
    df_mincer['Salario_efectivo_independiente'].fillna(0)
)

# ✅ Corrección obligatoria: usar df_mincer (en tu .py estaba usando df)
# --- FIX KeyError horas (mínimo) ---
# Buscar el nombre real de la columna de horas en el df original (por si varía)
col_horas = next((c for c in df.columns if "cuantas" in c.lower() and "horas" in c.lower()), None)

if col_horas is None:
    raise KeyError("No se encontró ninguna columna tipo 'Cuantas_horas_trabajó' en df. Revisa nombres en df.columns")

# Traer horas desde df original alineando por índice (df_mincer conserva índices del df original)
df_mincer['Cuantas_horas_trabajó'] = pd.to_numeric(df.loc[df_mincer.index, col_horas], errors='coerce')

# Continuar igual
df_mincer['horas_mensuales'] = df_mincer['Cuantas_horas_trabajó'] * 4.33
df_mincer['horas_mensuales'] = df_mincer['Cuantas_horas_trabajó'] * 4.33

df_mincer['experiencia'] = np.where(
    df_mincer['Edad'] >= 18,
    df_mincer['Edad'] - np.maximum(df_mincer['anios_educacion'] + 6, 18),
    np.nan
)
df_mincer['experiencia'] = df_mincer['experiencia'].clip(lower=0)
df_mincer['experiencia2'] = df_mincer['experiencia'] ** 2

df_mincer['salario_hora'] = df_mincer['ingreso_laboral'] / df_mincer['horas_mensuales']

df_mincer = df_mincer[
    (df_mincer['Edad'] >= 18) &
    (df_mincer['anios_educacion'].notna()) &
    (df_mincer['ingreso_laboral'] > 0) &
    (df_mincer['Cuantas_horas_trabajó'] > 0) &
    (df_mincer['horas_mensuales'] > 0)
].copy()

df_mincer['ln_ingreso'] = np.log(df_mincer['ingreso_laboral'])
df_mincer['ln_salario_hora'] = np.log(df_mincer['salario_hora'])

df_mincer = df_mincer.replace([np.inf, -np.inf], np.nan).dropna(subset=[
    'anios_educacion','experiencia','experiencia2','ln_ingreso'
])

X = sm.add_constant(df_mincer[['anios_educacion','experiencia','experiencia2']])
y = df_mincer['ln_ingreso']

modelo_mincer = sm.OLS(y, X).fit()
print(modelo_mincer.summary())

print("✅ DF_WAGE (df_mincer) shape:", df_mincer.shape)
df_mincer.head()

                            OLS Regression Results                            
Dep. Variable:             ln_ingreso   R-squared:                       0.298
Model:                            OLS   Adj. R-squared:                  0.298
Method:                 Least Squares   F-statistic:                     2465.
Date:                Sun, 24 May 2026   Prob (F-statistic):               0.00
Time:                        19:42:39   Log-Likelihood:                -24638.
No. Observations:               17410   AIC:                         4.928e+04
Df Residuals:                   17406   BIC:                         4.931e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               3.9031      0.031    1

,llave_sec,provincia,prov,unidad,cuest,hogar,Sexo,Edad,Jefe_de_hogar,Asiste_a_la_escuela,...,educacion_grupo,anios_educacion,ingreso_laboral,Cuantas_horas_trabajó,horas_mensuales,experiencia,experiencia2,salario_hora,ln_ingreso,ln_salario_hora
0,1.0,Bocas del Toro,01,001,01,1,Masculino,44.0,Jefe o jefa,2.0,...,Superior,16,1516.0,63.0,272.79,22.0,484.0,5.557388,7.323831,1.715128
1,1.0,Bocas del Toro,01,001,01,1,Femenino,21.0,Conyugue,2.0,...,Media,12,600.0,48.0,207.84,3.0,9.0,2.886836,6.396930,1.060161
3,2.0,Bocas del Toro,01,001,02,1,Femenino,22.0,Jefe o jefa,2.0,...,Media,12,700.0,48.0,207.84,4.0,16.0,3.367975,6.551080,1.214312
4,2.0,Bocas del Toro,01,001,02,1,Masculino,26.0,Conyugue,2.0,...,Media,12,850.0,66.0,285.78,8.0,64.0,2.974316,6.745236,1.090014
6,3.0,Bocas del Toro,01,001,03,1,Femenino,34.0,Jefe o jefa,2.0,...,Media,12,800.0,40.0,173.20,16.0,256.0,4.618938,6.684612,1.530165


Heckman 2 pasos: Probit + IMR + Outcome (df_heckman + df_wage)

In [22]:
df_heckman = df_modelo[columnas_finales].copy()
df_heckman.index = df_modelo.index

df_heckman['area_geografica'] = df['Área_geográfica']
df_heckman['prov'] = pd.to_numeric(df_heckman['prov'], errors='coerce').astype('Int64')

# Traer variables de Mincer a Heckman (si existen)
for col in ['ln_salario_hora','anios_educacion','experiencia','experiencia2']:
    if col in df_mincer.columns:
        df_heckman.loc[df_mincer.index, col] = df_mincer[col]

# Seguro social
df_heckman['Seguro_Social'] = df_modelo['Seguro_Social']
df_heckman['tiene_CSS'] = np.where(df_heckman['Seguro_Social'] == 6, 0, 1)

# ingreso_no_laboral (tal cual tu suma)
df['ingreso_no_laboral'] = (
    df['p72c1'].fillna(0) + df['p72c2'].fillna(0) + df['p72c3'].fillna(0) +
    df['p72c4'].fillna(0) + df['p72c5'].fillna(0) + df['p72c7'].fillna(0) +
    df['p72c8'].fillna(0) + df['p72f1'].fillna(0) + df['p72f2'].fillna(0) +
    df['p72f3'].fillna(0) + df['p72f4'].fillna(0)
)
df_heckman['ingreso_no_laboral'] = df['ingreso_no_laboral']

# Tipo de contrato
df_heckman['tipo_contrato'] = pd.to_numeric(df['Tipo_de_contrato'], errors='coerce').fillna(0)
df_heckman['tipo_contrato_eco'] = np.select(
    [df_heckman['tipo_contrato'].isin([1,4]),
     df_heckman['tipo_contrato'].isin([2,3]),
     df_heckman['tipo_contrato'] == 5],
    ['Formal','Temporal','Informal'],
    default='No especificado'
)

# Tamaño empresa (p29)
p29_clean = pd.to_numeric(df_modelo['p29'], errors='coerce')
df_heckman['tamano_empresa'] = p29_clean.map({
    1:'Microempresa',2:'Microempresa',3:'Pequeña',4:'Pequeña',5:'Mediana_Grande'
}).fillna('Sin respuesta')

# Provincia etiquetas
df_heckman['provincia'] = df_heckman['provincia'].astype(str).str.zfill(2).replace(dicc_provincia)

# Jornada (horas)
df_heckman['horas_semana'] = pd.to_numeric(df['Cuantas_horas_trabajó'], errors='coerce')
df_heckman['horas_mensuales'] = df_heckman['horas_semana'] * 4.33

def clasificar_jornada(h):
    if pd.isna(h): return np.nan
    if h < 15: return 'Muy reducida'
    if h < 40: return 'Parcial'
    if h <= 48: return 'Completa estándar'
    return 'Extendida'

df_heckman['jornada_categoria'] = df_heckman['horas_semana'].apply(clasificar_jornada)

# Estado civil recodificado (tal cual tu np.select)
df_heckman['Estado_civil'] = df['Estado_civil']
df_heckman['estado_civil_rec'] = np.select(
    [df_heckman['Estado_civil'].isin([1,4]),
     df_heckman['Estado_civil'].isin([2,3,6]),
     df_heckman['Estado_civil'] == 7,
     df_heckman['Estado_civil'] == 5,
     df_heckman['Estado_civil'] == 8],
    [1,2,3,4,5],
    default=np.nan
)

df_heckman['jefe_hogar_rec'] = np.select(
    [df_heckman['Jefe_de_hogar'] == 1,
     df_heckman['Jefe_de_hogar'] == 2,
     df_heckman['Jefe_de_hogar'] == 3,
     df_heckman['Jefe_de_hogar'] == 4,
     df_heckman['Jefe_de_hogar'].isin([5,6])],
    [1,2,3,4,5],
    default=np.nan
)

# Antigüedad
df['antiguedad_meses'] = pd.to_numeric(df['p34'], errors='coerce')
df['antiguedad_anios'] = df['antiguedad_meses'] / 12
df_heckman['antiguedad_anios'] = df['antiguedad_anios']

# Variable de selección
df_heckman['participa'] = np.where(df_heckman['ocu_des'] == 'Ocupados', 1, 0)

# ✅ Corrección obligatoria: NO crear columna vacía '' (en tu .py existía)
df_heckman['Edad2'] = df_heckman['Edad'] ** 2

# Probit
vars_seleccion = [
    'Edad','Edad2','Sexo','Asiste_a_la_escuela','Recibe_jubilacion_pension',
    'niños','adultos_mayores','ingreso_hogar','estado_civil_rec','jefe_hogar_rec',
    'ingreso_no_laboral','prov','participa'
]
df_sel = df_heckman[vars_seleccion].dropna().copy()

X_sel = df_sel[['Edad','Edad2','Sexo','Asiste_a_la_escuela','Recibe_jubilacion_pension',
                'niños','adultos_mayores','ingreso_hogar','estado_civil_rec','jefe_hogar_rec',
                'ingreso_no_laboral']]
X_sel = sm.add_constant(X_sel)
y_sel = df_sel['participa']

probit_res = sm.Probit(y_sel, X_sel).fit(disp=False)
print(probit_res.summary())

# IMR
df_sel['xb'] = probit_res.predict(X_sel, linear=True)
df_sel['IMR'] = norm.pdf(df_sel['xb']) / norm.cdf(df_sel['xb'])
df_sel['IMR'] = df_sel['IMR'].replace([np.inf, -np.inf], np.nan)

df_heckman = df_heckman.merge(df_sel[['IMR']], left_index=True, right_index=True, how='left')

# Outcome (salario) — usa ln_salario_hora si existe
vars_salario = [
    'ln_salario_hora','anios_educacion','experiencia','experiencia2','Sexo',
    'antiguedad_anios','IMR','tiene_CSS','horas_mensuales',
    'educacion_grupo','tipo_contrato_eco','tamano_empresa','area_geografica',
    'jornada_categoria','estado_civil_rec','provincia','participa'
]
vars_ok = [c for c in vars_salario if c in df_heckman.columns]
df_wage = df_heckman[vars_ok].copy()
df_wage = df_wage[df_wage['participa'] == 1].copy()

# Dummies categóricas
cols_dummies = [c for c in ['educacion_grupo','estado_civil_rec','tipo_contrato_eco','tamano_empresa',
                           'jornada_categoria','area_geografica','provincia'] if c in df_wage.columns]
if cols_dummies:
    df_wage = pd.get_dummies(df_wage, columns=cols_dummies, drop_first=True, dtype=int)

# Sexo a numérico si viene como texto
if 'Sexo' in df_wage.columns and df_wage['Sexo'].dtype == 'object':
    df_wage['Sexo'] = df_wage['Sexo'].replace({'Masculino':1,'Femenino':0,'Hombre':1,'Mujer':0})

# Forzar numérico
for c in df_wage.columns:
    df_wage[c] = pd.to_numeric(df_wage[c], errors='coerce')
df_wage = df_wage.dropna().copy()

# Modelo outcome si existe ln_salario_hora
if 'ln_salario_hora' in df_wage.columns:
    y_wage = df_wage['ln_salario_hora'].astype(float)
    X_wage = df_wage.drop(columns=['ln_salario_hora','participa'], errors='ignore')
    X_wage = sm.add_constant(X_wage).astype(float)
    wage_model = sm.OLS(y_wage, X_wage).fit()
    print(wage_model.summary())
else:
    wage_model = None
    print("⚠️ No existe ln_salario_hora; se omite modelo outcome.")

print("✅ DF_LABOR (df_heckman) shape:", df_heckman.shape)
print("✅ DF_WAGE Heckman outcome (df_wage) shape:", df_wage.shape)

                          Probit Regression Results                           
Dep. Variable:              participa   No. Observations:                40096
Model:                         Probit   Df Residuals:                    40084
Method:                           MLE   Df Model:                           11
Date:                Sun, 24 May 2026   Pseudo R-squ.:                  0.3998
Time:                        19:44:29   Log-Likelihood:                -16682.
converged:                       True   LL-Null:                       -27792.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -1.7353      0.080    -21.721      0.000      -1.892      -1.579
Edad                          0.1418      0.003     53.308      0.000       0.137     

vExportar TODO para Power BI (CSV + Excel + ZIP)

In [23]:
# A) Datasets para Power BI
df_modelo_final.to_csv(os.path.join(OUT_DIR, "DF_POP_desigualdad.csv"), index=False)
df_mincer.to_csv(os.path.join(OUT_DIR, "DF_WAGE_mincer.csv"), index=False)
df_heckman.to_csv(os.path.join(OUT_DIR, "DF_LABOR_heckman_base.csv"), index=False)
df_wage.to_csv(os.path.join(OUT_DIR, "DF_WAGE_heckman_outcome.csv"), index=False)

# B) Coeficientes para coefplots
mincer_tbl = modelo_mincer.summary2().tables[1].reset_index().rename(columns={'index':'variable'})
mincer_tbl.to_csv(os.path.join(OUT_DIR, "mincer_coef.csv"), index=False)

probit_tbl = probit_res.summary2().tables[1].reset_index().rename(columns={'index':'variable'})
probit_tbl.to_csv(os.path.join(OUT_DIR, "heckman_probit_coef.csv"), index=False)

if wage_model is not None:
    wage_tbl = wage_model.summary2().tables[1].reset_index().rename(columns={'index':'variable'})
    wage_tbl.to_csv(os.path.join(OUT_DIR, "heckman_outcome_coef.csv"), index=False)

# C) Excel multi-hoja
excel_path = os.path.join(OUT_DIR, "PowerBI_datasets.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_modelo_final.to_excel(writer, sheet_name="DF_POP_desigualdad", index=False)
    df_mincer.to_excel(writer, sheet_name="DF_WAGE_mincer", index=False)
    df_heckman.to_excel(writer, sheet_name="DF_LABOR_heckman_base", index=False)
    df_wage.to_excel(writer, sheet_name="DF_WAGE_heckman_outcome", index=False)
    mincer_tbl.to_excel(writer, sheet_name="mincer_coef", index=False)
    probit_tbl.to_excel(writer, sheet_name="heckman_probit_coef", index=False)
    if wage_model is not None:
        wage_tbl.to_excel(writer, sheet_name="heckman_outcome_coef", index=False)

print("✅ Exportado todo en:", OUT_DIR)
print(os.listdir(OUT_DIR))

# D) ZIP para descargar
zip_name = "PowerBI_outputs.zip"
zip_path = os.path.join(OUT_DIR, zip_name)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT_DIR):
        if fn != zip_name:
            z.write(os.path.join(OUT_DIR, fn), arcname=fn)

print("✅ ZIP listo:", zip_path)

from google.colab import files
files.download(zip_path)

✅ Exportado todo en: outputs_powerbi
['DF_WAGE_mincer.csv', 'heckman_outcome_coef.csv', 'PowerBI_datasets.xlsx', 'DF_LABOR_heckman_base.csv', 'PowerBI_outputs.zip', 'DF_POP_desigualdad.csv', 'heckman_probit_coef.csv', 'DF_WAGE_heckman_outcome.csv', 'mincer_coef.csv']
✅ ZIP listo: outputs_powerbi/PowerBI_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>